# Paper 05 · word2vec

**Citation:** Tomas Mikolov et al., “Efficient Estimation of Word Representations in Vector Space” (2013).

**Paper:** https://arxiv.org/abs/1301.3781

> **Scale gap:** We train skip-gram embeddings on a tiny handcrafted corpus. Semantic quality is limited by corpus scale.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. What is the training signal if there are no manual semantic labels?
2. How does context-window size change what 'similarity' means?
3. Why should tiny-corpus analogies be treated cautiously?

## Central claim
Predicting local context with a shallow neural objective can produce useful distributed word representations efficiently.

## Build skip-gram pairs

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-05_word2vec', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/05_word2vec.ipynb')
experiment.capture_figures()

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from sklearn.decomposition import PCA
corpus=("king queen prince princess royal palace "
        "man woman boy girl family "
        "king prince royal palace queen princess "
        "dog cat animal pet dog puppy cat kitten "
        "man king woman queen boy prince girl princess").split()
vocab=sorted(set(corpus)); stoi={w:i for i,w in enumerate(vocab)}
pairs=[]; window=2
for i,w in enumerate(corpus):
    for j in range(max(0,i-window),min(len(corpus),i+window+1)):
        if i!=j: pairs.append((stoi[w],stoi[corpus[j]]))
x=torch.tensor([p[0] for p in pairs]); y=torch.tensor([p[1] for p in pairs])
print("vocab",len(vocab),"training pairs",len(pairs))

## Partially completed skip-gram model

In [ ]:
class SkipGram(nn.Module):
    def __init__(self,V,D=8):
        super().__init__()
        self.emb=nn.Embedding(V,D)
        self.out=nn.Linear(D,V,bias=False)
    def forward(self,ids):
        # TODO: explain why the output is a distribution over vocabulary items.
        return self.out(self.emb(ids))
m=SkipGram(len(vocab),8); opt=torch.optim.Adam(m.parameters(),lr=.05); ce=nn.CrossEntropyLoss()
for _ in range(500):
    opt.zero_grad(); loss=ce(m(x),y); loss.backward(); opt.step()
print("loss",float(loss))

## Figure-inspired embedding map

In [ ]:
E=m.emb.weight.detach().numpy()
Z=PCA(2).fit_transform(E)
plt.figure(figsize=(8,6)); plt.scatter(Z[:,0],Z[:,1])
for word,(a,b) in zip(vocab,Z): plt.text(a,b,word)
plt.title("Tiny skip-gram embedding space"); plt.show()

def nearest(word,k=5):
    e=E[stoi[word]]; sim=E@e/(np.linalg.norm(E,axis=1)*np.linalg.norm(e)+1e-12)
    ids=np.argsort(sim)[::-1][1:k+1]
    return [(vocab[i],float(sim[i])) for i in ids]
print("king:",nearest("king"))
print("dog:",nearest("dog"))

### Ablation
Retrain with window sizes 1 and 5. Record nearest neighbors and explain what kind of relationships become easier or harder to capture.

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))